# 01 - Spatial Temporal Error Diagnostic
Diagnostico espacial, temporal y de calibracion para los 6 modelos campeones (retornos + clasificacion calibrada).

In [1]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, accuracy_score
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from xgboost import XGBRegressor, XGBClassifier

base_dir = Path.cwd()
for root in [base_dir, *base_dir.parents]:
    if (root / "src").exists():
        if str(root) not in sys.path:
            sys.path.insert(0, str(root))
        break

from src.data_processing.build_dataset import get_training_features

warnings.filterwarnings("ignore")

dataset_path = None
for root in [base_dir, *base_dir.parents]:
    candidate = root / "data" / "processed" / "dataset_entrenamiento_final.csv"
    if candidate.exists():
        dataset_path = candidate
        break
if dataset_path is None:
    raise FileNotFoundError("dataset_entrenamiento_final.csv not found under data/processed")

df = pd.read_csv(dataset_path, parse_dates=["date"])
print("dataset:", dataset_path.resolve())
print("shape:", df.shape)

blacklist = [
    "precio_provincial_lag_1"
    "precio_provincial_lag_2"
    "precio_provincial_lag_3"
    "precio_vecinos_media_lag1"
    "precio_nacional_base_ma3"
    "precio_nacional_base_ma6"
    "precio_nacional_base_vol3"
    "precio_nacional_base_vol6"
]
target_candidates = ["precio_provincial_TARGET_H1", "precio_provincial_TARGET_H2", "precio_provincial_TARGET_H3"]
missing_targets = [t for t in target_candidates if t not in df.columns]
if missing_targets:
    raise ValueError(f"Missing target columns: {missing_targets}")
available_targets = target_candidates
horizons = [1, 2, 3]

split_date = pd.Timestamp("2021-01-01")
train_mask = df["date"] < split_date
test_mask = ~train_mask

train_df = df.loc[train_mask].copy()
test_df = df.loc[test_mask].copy()
print("train rows:", train_df.shape[0], "test rows:", test_df.shape[0])

identifiers = ["date", "provincia", "cereal_predominante"]
training_cols = get_training_features(df)
feature_cols = [
    c for c in training_cols
    if c in df.columns and c not in identifiers + available_targets
]
feature_cols = [c for c in feature_cols if c not in blacklist]

X_full = df[feature_cols].copy()
bool_cols = X_full.select_dtypes(include=["bool"]).columns
if len(bool_cols) > 0:
    X_full[bool_cols] = X_full[bool_cols].astype(int)

cat_cols = X_full.select_dtypes(include=["object", "category"]).columns.tolist()
if cat_cols:
    X_full = pd.get_dummies(X_full, columns=cat_cols, drop_first=False)

X_train = X_full.loc[train_mask].copy()
X_test = X_full.loc[test_mask].copy()
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

base_price_col = "precio_provincial_lag_1"
if base_price_col not in df.columns:
    raise ValueError("precio_provincial_lag_1 missing for targets")

def build_targets(horizon: int):
    target_reg = f"precio_provincial_TARGET_H{horizon}"
    y_train_reg = train_df[target_reg]
    y_test_reg = test_df[target_reg]
    base_train = train_df[base_price_col]
    base_test = test_df[base_price_col]
    y_train_clf = (y_train_reg - base_train > 0).astype(int)
    y_test_clf = (y_test_reg - base_test > 0).astype(int)
    return y_train_reg, y_test_reg, y_train_clf, y_test_clf, base_train, base_test

def regression_metrics(y_true, y_pred):
    aligned = pd.concat([y_true, y_pred], axis=1).dropna()
    if aligned.empty:
        return {"MAE": np.nan, "RMSE": np.nan, "Pearson": np.nan}
    y_true_clean = aligned.iloc[:, 0]
    y_pred_clean = aligned.iloc[:, 1]
    mae = mean_absolute_error(y_true_clean, y_pred_clean)
    rmse = np.sqrt(mean_squared_error(y_true_clean, y_pred_clean))
    pearson = pearsonr(y_true_clean, y_pred_clean)[0] if y_true_clean.nunique() > 1 else np.nan
    return {"MAE": float(mae), "RMSE": float(rmse), "Pearson": float(pearson) if pearson == pearson else np.nan}

def classification_metrics(y_true, proba, pred):
    acc = accuracy_score(y_true, pred)
    return {"DA": float(acc)}

def top_features_from_model(model, feature_names, top_k=5):
    if hasattr(model, "feature_importances_"):
        importances = model.feature_importances_
        order = np.argsort(importances)[::-1][:top_k]
        return [feature_names[i] for i in order]
    if hasattr(model, "coef_"):
        coefs = np.ravel(model.coef_)
        order = np.argsort(np.abs(coefs))[::-1][:top_k]
        return [feature_names[i] for i in order]
    return []

dataset: C:\Users\marco\Desktop\Repos\DATAGIA-21\data\processed\dataset_entrenamiento_final.csv
shape: (7047, 78)
train rows: 5394 test rows: 1653


## 1. Entrenamiento de modelos campeones V1
Regresion por retornos y clasificacion calibrada para H1-H3.

In [3]:
tscv = TimeSeriesSplit(n_splits=5)

reg_param_grids = {
    "Ridge": {"model__alpha": [0.1, 1.0, 5.0, 10.0, 25.0]},
    "RF": {
        "model__n_estimators": [300, 500, 700],
        "model__max_depth": [4, 6, 8, 12, None],
        "model__min_samples_leaf": [1, 2, 4],
        "model__max_features": ["sqrt", 0.6, 0.8],
    },
    "XGB": {
        "model__n_estimators": [200, 400, 600],
        "model__max_depth": [3, 5, 7],
        "model__learning_rate": [0.01, 0.05, 0.1],
        "model__subsample": [0.6, 0.8, 1.0],
        "model__colsample_bytree": [0.6, 0.8, 1.0],
        "model__min_child_weight": [1, 5, 10],
    },
}

clf_param_grids = {
    "RF": {
        "model__n_estimators": [300, 500, 700],
        "model__max_depth": [4, 6, 8, 12, None],
        "model__min_samples_leaf": [1, 2, 4],
        "model__max_features": ["sqrt", 0.6, 0.8],
    },
    "XGB": {
        "model__n_estimators": [200, 400, 600],
        "model__max_depth": [3, 5, 7],
        "model__learning_rate": [0.01, 0.05, 0.1],
        "model__subsample": [0.6, 0.8, 1.0],
        "model__colsample_bytree": [0.6, 0.8, 1.0],
        "model__min_child_weight": [1, 5, 10],
    },
}

return_champions = {}
clf_calibrated = {}
return_top5 = {}

for h in horizons:
    y_train_reg, y_test_reg, y_train_clf, y_test_clf, base_train, base_test = build_targets(h)
    train_mask_h = base_train.notna() & (base_train != 0) & y_train_reg.notna()
    test_mask_h = base_test.notna() & (base_test != 0) & y_test_reg.notna()
    X_train_h = X_train.loc[train_mask_h]
    X_test_h = X_test.loc[test_mask_h]
    base_train_h = base_train.loc[train_mask_h]
    base_test_h = base_test.loc[test_mask_h]

    y_train_ret = (y_train_reg.loc[train_mask_h] - base_train_h) / base_train_h
    y_test_ret = (y_test_reg.loc[test_mask_h] - base_test_h) / base_test_h

    reg_models = {
        "Ridge": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", Ridge(random_state=42)),
        ]),
        "RF": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", RandomForestRegressor(random_state=42, n_jobs=-1)),
        ]),
        "XGB": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", XGBRegressor(random_state=42, n_jobs=-1, objective="reg:squarederror")),
        ]),
    }

    best_by_name = {}
    for name, model in reg_models.items():
        search = RandomizedSearchCV(
            model,
            param_distributions=reg_param_grids[name],
            n_iter=20,
            scoring="neg_mean_absolute_error",
            cv=tscv,
            random_state=42,
            n_jobs=-1,
        )
        search.fit(X_train_h, y_train_ret)
        best_by_name[name] = search.best_estimator_

    best_name = None
    best_metrics = None
    best_model = None
    for name, model in best_by_name.items():
        preds = pd.Series(model.predict(X_test_h), index=y_test_ret.index)
        metrics = regression_metrics(y_test_ret, preds)
        if best_metrics is None or metrics["MAE"] < best_metrics["MAE"]:
            best_name = name
            best_metrics = metrics
            best_model = model

    return_champions[h] = {
        "model": best_name,
        "metrics": best_metrics,
        "estimator": best_model,
        "y_test_ret": y_test_ret,
        "X_test": X_test_h,
    }
    return_top5[h] = top_features_from_model(best_model.named_steps["model"], X_train.columns.tolist(), top_k=5)

    train_mask_c = y_train_clf.notna()
    test_mask_c = y_test_clf.notna()
    X_train_c = X_train.loc[train_mask_c]
    X_test_c = X_test.loc[test_mask_c]
    y_train_c = y_train_clf.loc[train_mask_c]
    y_test_c = y_test_clf.loc[test_mask_c]

    clf_models = {
        "RF": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", RandomForestClassifier(random_state=42, n_jobs=-1)),
        ]),
        "XGB": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", XGBClassifier(random_state=42, n_jobs=-1, eval_metric="logloss")),
        ]),
    }

    best_by_name = {}
    for name, model in clf_models.items():
        search = RandomizedSearchCV(
            model,
            param_distributions=clf_param_grids[name],
            n_iter=20,
            scoring="roc_auc",
            cv=tscv,
            random_state=42,
            n_jobs=-1,
        )
        search.fit(X_train_c, y_train_c)
        best_by_name[name] = search.best_estimator_

    best_name = None
    best_metrics = None
    best_model = None
    for name, model in best_by_name.items():
        proba = model.predict_proba(X_test_c)[:, 1]
        pred = (proba >= 0.5).astype(int)
        metrics = classification_metrics(y_test_c, proba, pred)
        if best_metrics is None or metrics["DA"] > best_metrics["DA"]:
            best_name = name
            best_metrics = metrics
            best_model = model

    calib = CalibratedClassifierCV(estimator=best_model, method="sigmoid", cv=TimeSeriesSplit(n_splits=3))
    calib.fit(X_train_c, y_train_c)
    proba_cal = calib.predict_proba(X_test_c)[:, 1]
    pred_cal = (proba_cal >= 0.5).astype(int)

    clf_calibrated[h] = {
        "model": best_name,
        "estimator": calib,
        "y_test": y_test_c,
        "X_test": X_test_c,
        "proba": pd.Series(proba_cal, index=y_test_c.index),
        "pred": pd.Series(pred_cal, index=y_test_c.index),
    }

print("Return champions:", {h: return_champions[h]["model"] for h in horizons})
print("Clf calibrated ready for horizons:", list(clf_calibrated.keys()))

Return champions: {1: 'RF', 2: 'RF', 3: 'RF'}
Clf calibrated ready for horizons: [1, 2, 3]


## 2. Analisis espacial (errores por provincia)
MAE y DA por provincia para cada horizonte.

In [5]:
def top_bottom(df_in, metric, n=5):
    df_sorted = df_in.sort_values(metric)
    return df_sorted.head(n), df_sorted.tail(n)

spatial_tables = {}

for h in horizons:
    ret = return_champions[h]
    y_test_ret = ret["y_test_ret"]
    X_test_h = ret["X_test"]
    pred_ret = pd.Series(ret["estimator"].predict(X_test_h), index=y_test_ret.index)
    ret_df = pd.DataFrame({
        "provincia": test_df.loc[y_test_ret.index, "provincia"],
        "y_true": y_test_ret,
        "y_pred": pred_ret,
    })
    ret_df["abs_err"] = (ret_df["y_true"] - ret_df["y_pred"]).abs()
    ret_df["da"] = (np.sign(ret_df["y_true"]) == np.sign(ret_df["y_pred"]))

    reg_by_prov = ret_df.groupby("provincia").agg(MAE=("abs_err", "mean"), DA=("da", "mean")).reset_index()

    clf = clf_calibrated[h]
    clf_df = pd.DataFrame({
        "provincia": test_df.loc[clf["y_test"].index, "provincia"],
        "y_true": clf["y_test"],
        "y_pred": clf["pred"],
    })
    clf_df["da"] = (clf_df["y_true"] == clf_df["y_pred"])
    clf_by_prov = clf_df.groupby("provincia").agg(DA=("da", "mean")).reset_index()

    spatial_tables[h] = {
        "reg_by_prov": reg_by_prov,
        "clf_by_prov": clf_by_prov,
        "reg_best": top_bottom(reg_by_prov, "MAE")[0],
        "reg_worst": top_bottom(reg_by_prov, "MAE")[1],
        "clf_best": top_bottom(clf_by_prov, "DA")[1],
        "clf_worst": top_bottom(clf_by_prov, "DA")[0],
    }

spatial_tables[1]["reg_best"], spatial_tables[1]["reg_worst"], spatial_tables[1]["clf_best"], spatial_tables[1]["clf_worst"]

(     provincia       MAE        DA
 18  Valladolid  0.041785  0.745614
 19      Zamora  0.042579  0.754386
 5        Cádiz  0.043032  0.736842
 14     Sevilla  0.043853  0.719298
 13     Segovia  0.044000  0.736842,
      provincia       MAE        DA
 3  Ciudad Real  0.049817  0.719298
 8       Huesca  0.050804  0.666667
 0     Albacete  0.050946  0.736842
 9       Lleida  0.050961  0.666667
 4       Cuenca  0.051817  0.754386,
       provincia        DA
 16       Teruel  0.736842
 11     Palencia  0.745614
 20     Zaragoza  0.745614
 4        Cuenca  0.754386
 7   Guadalajara  0.789474,
     provincia        DA
 10    Navarra  0.657895
 1     Badajoz  0.666667
 9      Lleida  0.684211
 6     Córdoba  0.684211
 12  Salamanca  0.684211)

## 3. Analisis temporal (crisis 2022-2023)
Residuos por mes y cruce con variables de trigo y urea.

In [7]:
def pick_feature(df_in, candidates):
    for name in candidates:
        if name in df_in.columns:
            return name
    return None

wheat_col = pick_feature(df, ["wheat_intl_eur_ma3", "wheat_intl_eur_lag_1", "wheat_intl_eur"])
urea_col = pick_feature(df, ["prepag1_urea 46_lag_1", "prepag1_urea 46_lag_2", "prepag1_urea 46"])

temporal_rows = []

for h in horizons:
    ret = return_champions[h]
    y_test_ret = ret["y_test_ret"]
    X_test_h = ret["X_test"]
    pred_ret = pd.Series(ret["estimator"].predict(X_test_h), index=y_test_ret.index)
    err = (y_test_ret - pred_ret).abs()
    err_df = pd.DataFrame({
        "date": test_df.loc[y_test_ret.index, "date"],
        "abs_err": err,
    })
    err_df["month"] = err_df["date"].dt.to_period("M").astype(str)
    monthly = err_df.groupby("month")["abs_err"].sum().reset_index()
    monthly = monthly.sort_values("abs_err", ascending=False).head(3)

    for _, row in monthly.iterrows():
        month = row["month"]
        month_mask = test_df["date"].dt.to_period("M").astype(str) == month
        wheat_val = float(test_df.loc[month_mask, wheat_col].mean()) if wheat_col else np.nan
        urea_val = float(test_df.loc[month_mask, urea_col].mean()) if urea_col else np.nan
        temporal_rows.append({
            "horizon": h,
            "month": month,
            "abs_err_sum": float(row["abs_err"]),
            "wheat_mean": wheat_val,
            "urea_mean": urea_val,
        })

temporal_df = pd.DataFrame(temporal_rows)
temporal_df

,horizon,month,abs_err_sum,wheat_mean,urea_mean
0,1,2022-03,6.376351,962.209432,90.71
1,1,2021-10,5.548165,645.643748,48.44
2,1,2022-04,5.110469,990.655635,95.42
3,2,2022-03,8.605686,962.209432,90.71
4,2,2022-02,7.327880,782.425284,86.63
5,2,2021-09,5.820682,612.116065,44.22
6,3,2022-02,9.678424,782.425284,86.63
7,3,2022-03,8.992931,962.209432,90.71
8,3,2021-08,8.160952,606.190544,41.79


## 5. Reporte
Genera DIAGNOSTICO_ERRORES_V1.md con hallazgos clave.

In [9]:
report_root = None
for root in [base_dir, *base_dir.parents]:
    candidate = root / "reports"
    if candidate.exists():
        report_root = candidate
        break
if report_root is None:
    report_root = base_dir / "reports"
    report_root.mkdir(parents=True, exist_ok=True)

report_path = report_root / "DIAGNOSTICO_ERRORES_V1.md"

lines = [
    "# DIAGNOSTICO_ERRORES_V1",
    "",
    "## Resumen",
    "Diagnostico espacial, temporal y de calibracion para modelos V1.",
    "",
    "## Analisis espacial (top/bottom por provincia)",
]

for h in horizons:
    lines.append(f"### Horizonte H{h}")
    lines.append("Regresion (MAE) - mejores 5:")
    lines.append(spatial_tables[h]["reg_best"].to_markdown(index=False))
    lines.append("")
    lines.append("Regresion (MAE) - peores 5:")
    lines.append(spatial_tables[h]["reg_worst"].to_markdown(index=False))
    lines.append("")
    lines.append("Clasificacion (DA) - mejores 5:")
    lines.append(spatial_tables[h]["clf_best"].to_markdown(index=False))
    lines.append("")
    lines.append("Clasificacion (DA) - peores 5:")
    lines.append(spatial_tables[h]["clf_worst"].to_markdown(index=False))
    lines.append("")

top5_df = pd.DataFrame([{
    "horizon": h,
    "Top5": ", ".join(return_top5[h]),
} for h in horizons])

lines.extend([
    "## Analisis temporal (top 3 meses por error acumulado)",
    temporal_df.to_markdown(index=False),
    "",
    "## Reliability curves",
    "Revisar diagramas de calibracion en el notebook.",
    "",
    "## Top 5 variables (retornos)",
    top5_df.to_markdown(index=False),
    "",
])

report_path.write_text("\n".join(lines), encoding="utf-8")
print("Reporte guardado en:", report_path.resolve())

Reporte guardado en: C:\Users\marco\Desktop\Repos\DATAGIA-21\reports\DIAGNOSTICO_ERRORES_V1.md
